In [2]:
import secrets, hashlib

def gen_label():
    # 16 random bytes as a wire label
    return secrets.token_bytes(16)

def H(key, msg):
    return hashlib.sha256(key + msg).digest()

def build_garbled_and():
    # Wire labels: (0-label, 1-label) for A, B, and OUT
    A0, A1 = gen_label(), gen_label()
    B0, B1 = gen_label(), gen_label()
    O0, O1 = gen_label(), gen_label()

    # Truth table for AND
    # (A,B) -> OUT
    truth = {
        (0, 0): O0,
        (0, 1): O0,
        (1, 0): O0,
        (1, 1): O1,
    }

    # Map bits to labels (for input owners)
    input_labels = {
        "A": {0: A0, 1: A1},
        "B": {0: B0, 1: B1},
    }

    # Garbled table: each row is an encryption of the correct output label
    # using the two input labels as keys.
    garbled_table = []
    for a_bit, a_lab in [(0, A0), (1, A1)]:
        for b_bit, b_lab in [(0, B0), (1, B1)]:
            out_lab = truth[(a_bit, b_bit)]
            pad = H(a_lab, b_lab)
            cipher = bytes(o ^ p for o, p in zip(out_lab, pad))
            garbled_table.append((a_lab, b_lab, cipher))

    # Evaluator only needs the ciphers, not the (a_lab, b_lab) here.
    # For a clearer demo we’ll keep full tuples, then filter using actual labels.
    out_label_to_bit = {O0: 0, O1: 1}

    return input_labels, garbled_table, out_label_to_bit

def evaluate_and(garbled_table, labelA, labelB, out_label_to_bit):
    # Nancy has labelA (from Omar’s input A) and labelB (her own B)
    for a_lab, b_lab, cipher in garbled_table:
        if a_lab == labelA and b_lab == labelB:
            pad = H(labelA, labelB)
            out_lab = bytes(c ^ p for c, p in zip(cipher, pad))
            return out_label_to_bit[out_lab]
    raise ValueError("No matching row found")

# ==== Demo ====
if __name__ == "__main__":
    # Garbler sets up circuit
    input_labels, garbled_table, out_map = build_garbled_and()

    # Omar chooses A
    A_bit = 1   # Omar's private bit
    A_label = input_labels["A"][A_bit]  # sent to Nancy via OT in real protocol

    # Nancy chooses B
    B_bit = 0   # Nancy's private bit
    B_label = input_labels["B"][B_bit]

    # Nancy evaluates garbled AND without learning A_bit
    result = evaluate_and(garbled_table, A_label, B_label, out_map)
    print(f"A AND B = {result}")


A AND B = 0
